# Lev encoder decision model — Kaggle smoke run

Trains a **ModernBERT** backbone with a two-way `false / true` head on a frozen
Lev pack, then scores it with the unchanged `lev report`.

## Before you run

1. **Settings → Internet → On.** The backbone is pulled from the Hub by commit SHA.
2. **Settings → Accelerator → GPU T4 x2** (or P100). One card is enough; the
   second is unused because this path is single-device.
3. **The frozen pack is already attached** as the `saltbo/lev-en-joint-002`
   Dataset, mounted at `/kaggle/input/lev-en-joint-002`. If you swap packs,
   update `DATASET` below and add the new source to the kernel metadata. The
   pack must contain `manifest.json`, `rubric.json`, `records.jsonl` and the
   four split files; the first cell refuses anything else.

## What this does and does not prove

It proves the encoder path trains, reloads and scores on this hardware, and what
the development accuracy and wall-clock actually are. It does **not** replace a
test-set reading, an independent calibration, or a release decision: pick and
freeze the model on development, then read test once.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
    capture_output=True, text=True,
).stdout if shutil.which("nvidia-smi") else
      "no nvidia-smi on PATH: this session has no GPU accelerator")

In [ ]:
REPO = "https://github.com/eslsoft/lev.git"
BRANCH = "experiment/plan-a-modernbert"
CONFIG = "configs/encoder-modernbert-base.toml"
TRAIN_RUN = "enc-train-001"
DEV_RUN = "enc-dev-001"

WORK = Path("/kaggle/working")
DATASET = Path("/kaggle/input/lev-en-joint-002")
SRC = WORK / "lev"

required = {"manifest.json", "rubric.json", "records.jsonl"} | {
    f"{split}.jsonl" for split in ("train", "development", "calibration", "test")
}
missing = sorted(required - {p.name for p in DATASET.glob("*")})
if missing:
    raise SystemExit(
        f"{DATASET} is not a frozen pack; missing {missing}.\n"
        "Upload the pack as a Kaggle Dataset and set DATASET to its directory."
    )
print("frozen pack:", DATASET)
print("output:", WORK)

In [ ]:
if SRC.exists():
    shutil.rmtree(SRC)
subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(SRC)], check=True
)

# --no-deps keeps Kaggle's own torch; only pull the extra if the import fails.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(SRC)],
    check=True,
)

ENV = os.environ.copy()
ENV["PYTHONPATH"] = str(SRC / "src")
# Bucketed batches keep padding small; this keeps the allocator from fragmenting.
ENV.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


def lev(*args):
    print("$ lev", " ".join(args), flush=True)
    subprocess.run([sys.executable, "-m", "lev", *args], cwd=SRC, env=ENV, check=True)


lev("check-upstream", "--help")  # cheap import check for the CLI itself

try:
    import torch
    import transformers
    if tuple(int(x) for x in transformers.__version__.split(".")[:2]) < (4, 48):
        raise ImportError(f"transformers {transformers.__version__} predates ModernBERT")
except ImportError as error:
    print(f"installing the encoder extra: {error}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"{SRC}[encoder]"], check=True
    )
    import torch
    import transformers

print("torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA device. Set Settings -> Accelerator -> GPU T4 x2 on this notebook "
        "and re-run; training 15,515 pairs on CPU would take hours."
    )
if torch.cuda.is_available():
    # T4 / P100 are sm_75 / sm_60: no flash kernel, so long sequences cost real memory.
    print(
        "sdpa flash", torch.backends.cuda.flash_sdp_enabled(),
        "| mem_efficient", torch.backends.cuda.mem_efficient_sdp_enabled(),
        "| math", torch.backends.cuda.math_sdp_enabled(),
    )
    print("capability", torch.cuda.get_device_capability(0))

## Train

`lev encode` refuses to overwrite an existing output directory, so a re-run
needs a new `TRAIN_RUN`. Over-long inputs are refused rather than truncated —
if this fails on `--max-len`, raise `max_len` in the config instead of cutting
the state.

Batches are built by sorting records by length and packing them up to
`max_tokens` **padded** tokens, so one long record never pads a whole batch of
short ones. `max_len` is the refusal line; `max_tokens` is the memory budget.
If you hit an OOM, lower `max_tokens` in the config — never `max_len`.
T4 (sm_75) and P100 (sm_60) have no flash-attention kernel, so this distinction
matters more there than on an A100.

In [ ]:
lev(
    "encode",
    "--dataset", str(DATASET),
    "--config", CONFIG,
    "--out", str(WORK / TRAIN_RUN),
    "--device", "cuda",
    "--execute",
)

trained = json.loads((WORK / TRAIN_RUN / "checkpoint" / "lev-encoder.json").read_text())
print("pairs:", trained["data"]["pairs"], "| total seconds:", trained["seconds"])
for epoch in trained["history"]:
    peak = epoch["peak_device_bytes"]
    print(
        f"epoch {epoch['epoch']}: loss {epoch['train_loss']:.4f} "
        f"{epoch['seconds']}s peak {peak / 2**30:.2f} GiB" if peak
        else f"epoch {epoch['epoch']}: loss {epoch['train_loss']:.4f} {epoch['seconds']}s"
    )

## Score development

The threshold is fixed at 0.5 here and is **not** tuned on this split's result.
If you later want a different threshold, pre-register it on development before
reading calibration or test.

In [ ]:
lev(
    "encode-benchmark",
    "--dataset", str(DATASET),
    "--config", CONFIG,
    "--checkpoint", str(WORK / TRAIN_RUN / "checkpoint"),
    "--split", "development",
    "--out", str(WORK / DEV_RUN),
    "--device", "cuda",
    "--execute",
)

lev(
    "report",
    "--benchmark", str(WORK / DEV_RUN),
    "--threshold", "0.5",
    "--out", str(WORK / DEV_RUN / "lev-report.json"),
)

report = json.loads((WORK / DEV_RUN / "lev-report.json").read_text())
d = report["decisions"]
print(f"engine: {report['engine']}")
print(f"split: {report['split']}  threshold: {report['threshold']}")
print(
    f"accuracy_all_requested {d['accuracy_all_requested']['rate']:.2%} "
    f"({d['accuracy_all_requested']['numerator']}/{d['accuracy_all_requested']['denominator']})"
)
print(f"completed {d['completed']}  missing {d['missing']}")
print(f"false_rejections {d['false_rejections']}  false_acceptances {d['false_acceptances']}")
print(f"brier_completed_only {d['brier_completed_only']}")
print()
print(f"{'kind':38} {'correct':>9} {'rate':>8}  missing")
for kind, m in report["by_kind"].items():
    rate = m["accuracy_all_requested"]["rate"]
    print(
        f"{kind:38} "
        f"{m['accuracy_all_requested']['numerator']:>4}/{m['accuracy_all_requested']['denominator']:<4} "
        f"{rate:>7.2%}  {m['missing']}"
    )

## Compare with the Kev 4B baseline

Reference points from this repository, on **different** packs and splits, so
treat them as scale, not as a paired comparison:

| model | params | development |
|---|---:|---:|
| raw Kev-4B | 4B | 364 / 590 (61.69%) |
| Lev v0.2 (4B + LoRA) | 4B | 483 / 590 (81.86%) |

A ModernBERT-base run on `en-joint-002` development is a **new** measurement on
a pack the 4B numbers were not taken on. Do not put them in the same table
without saying so.

## Before you close the notebook

Save Version writes `/kaggle/working` to the notebook's output. Without it, the
checkpoint and the report are discarded when the session ends.